# 🎯 [OPS04_M03] 속도·고도·접근 방위각 기반 1차 공중위협 우선순위 산정 알고리즘 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M03 |
| **미션 주제** | 속도·고도·접근 방위각 기반 1차 공중위협 우선순위 산정 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **핵심 산출물** | `src/models/ops04_baseline_model.py, 기본 추론 결과 보고서` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 속도·고도·접근 방위각 기반 1차 공중위협 우선순위 산정 알고리즘 구현
**소요시간**: 약 2.5시간 | **이번 산출물**: `src/models/ops04_baseline_model.py, 기본 추론 결과 보고서`

### 📋 미션 개요
**방어 자산까지의 최근접점(CPA)과 잔여 도달 예상시간(TTI)만을 이용해 1차 공중위협 우선순위를 산출하는 설명 가능한 Baseline 알고리즘을 구현합니다.**

- **배경 및 필요성:** 고도화된 다차원 평가 모델을 도입하기 전에, 물리적 도달시간 기반의 단순하고 투명한 기준선(Baseline)을 먼저 구축해야 이후 고도화 모델의 성능 향상과 기만체 오경보 한계를 객관적으로 비교할 수 있습니다.
- **핵심 과업:** 각 항적의 최근접점(CPA)이 가상 자산 반경 1.5 km 이내를 통과하는지 계산하고 TTI 순으로 정렬하며, TTI 600초 이내 표적을 '긴급'으로 판정해 정확도·상위 10개 일치율·유형별 오경보를 분석합니다.
- **학습 목표 (습득 역량):** 항적 벡터 기반 CPA 및 TTI 산출 기법, 기준선(Baseline) 랭킹 모델 설계 능력, 오탐·오경보의 원인 유형(기만체 등)을 정량적으로 진단하는 분석 역량을 체득합니다.
- **검증 및 산출물:** 기본 데이터셋 긴급 판정 정확도 75% 이상을 충족한 Baseline 모델(`src/models/ops04_baseline_model.py`)과 기본 추론 결과 보고서를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 비행체 항적의 기하학적 이동 벡터로부터 방어 자산까지의 최근접점(CPA)과 도달 예상시간(TTI)을 계산하고, 판단 근거가 투명한 Baseline 위협 우선순위 모델 설계 및 한계 진단 역량을 체득합니다.

**[핵심 과업]** 각 항적의 CPA가 가상 자산 반경 1.5 km 이내를 통과하는지 산출해 TTI 짧은 순으로 정렬하고, TTI 600초 이내를 '긴급'으로 판정해 실제 긴급 여부 대비 정확도·상위 10개 일치율·유형별 오경보를 분석합니다.

- **전술 기하 연산(CPA·TTI) 모델링 역량:** 표적의 위치·속도·접근 방위각으로부터 방어 거점 최근접 통과 거리와 잔여 도달 시간을 수학적으로 산출해 위협 임박도를 계량화하는 능력을 익힙니다.
- **설명 가능한 기준선(Baseline) 랭킹 설계 능력:** 복잡한 분류기 도입 전에 단순 물리 규칙(TTI 600초 기준)으로 1차 우선순위를 정렬해 비교 기준점과 판단 투명성을 확보하는 설계 원리를 습득합니다.
- **오경보 원인 유형별 진단 및 평가 역량:** 긴급 판정 정확도와 상위 10개 순위 일치율, 비행체 유형별 오경보 분포를 분석해 거리·시간만 고려할 때 기만체(decoy) 오판이 집중되는 한계를 규명하는 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
방어 거점까지의 거리, 잔여 도달 시간, 접근 속도를 바탕으로 1차 위협 순위를 정렬하는 Baseline 랭킹 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **방어 자산까지의 거리와 잔여 도달 예상시간(TTI)만으로 1차 위협 순위를 정렬하는 투명한 Baseline을 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **각 항적의 최근접점(CPA)이 가상 자산 반경 1.5 km 안을 지나는지 계산하고, 해당 자산까지의 도달 예상시간을 구해 짧은 순으로 정렬 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/ops04_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 3 코드를 실행해 상위 10개 위협 순위표를 확인한다.
#### 이 단계의 역할
미션 3의 `baseline_rank()` 알고리즘을 실행해 최근접점(CPA ≤ 1.5 km)과 도달 예상시간(TTI) 기준 상위 10개 위협 순위표를 확인합니다.

#### 해야 할 일

- 비행체의 현재 위치·속도·방위각 벡터로부터 가상 방어자산 4곳(`ASSETS`)까지의 최근접점(CPA)과 TTI 계산식을 확인한다.
- AI를 Co-pilot으로 활용해 CPA 반경 1.5 km(`CPA_KM`) 이내 진입 여부와 TTI 오름차순 정렬의 기하학적 근거를 검토한다.
- 상위 10개 위협 순위표와 긴급 판정 정확도(목표 75% 이상)를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "가상 방어자산 좌표(x, y)와 비행체의 위치·속도·방위각을 입력받아 최근접점(CPA 
#### 핵심 확인 지표

> 긴급 판정 정확도 90.0% (목표 75% 이상 달성), 상위 10개 위협 순위표 생성

#### 학습자가 제출해야 하는 내용

- 미션 3 실행 결과 및 상위 10개 위협 순위표 로그
- AI 프롬프트 질의 및 CPA·방위각 변환식 검증 기록

#### Step 2. 유형별 판정 결과에서 오경보가 어느 유형에서 나오는지 찾는다.
#### 이 단계의 역할
비행체 유형별(`uav_type`, `fast_type`, `decoy`, `civil`, `bird`) 판정 결과에서 오경보가 집중 발생하는 원인을 규명합니다.

#### 해야 할 일

- Baseline 판정 결과와 실제 위협 라벨을 교차 집계해 유형별 오경보·미탐 건수를 추출한다.
- AI에게 기만체(`decoy`)나 민항기/조류에서 오경보가 발생하는 물리적 이유를 데이터 수치 기반으로 분석해 달라고 요청한다.
- 전체 오경보 8건 중 기만체(`decoy` 7건)가 차지하는 비중과 거리·TTI 단일 기준의 한계를 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "Baseline 순위 모델의 유형별(uav_type, fast_type, decoy, civil, bird) 오경보 혼동행렬을 집계하고, 왜 기만체(decoy)에서 오경보가 집중 발생하는지 물리적·데이터 관점에서 원인을 분석해줘."

  - **AI 결과 검증 포인트:** AI가 분석한 오경보 건수가 실제 출력 로그(오경보 8건 중 decoy 7건)와 정확히 일치하는지 확인하고, 다음 미션(다차원 위협 평가)으로 넘어가야 할 근거가 명확한지 점검합니다.

#### 핵심 확인 지표

> 오경보 8건 분석 완료 (기만체 decoy 7건 집중 원인 규명)

#### 학습자가 제출해야 하는 내용

- 유형별 오경보 분포표 및 기만체 오판 원인 분석 메모
- AI 분석 결과 대조 검증 기록

#### Step 3. 그래프에서 도달시간 600초 기준선과 실제 긴급 표적을 비교한다.
#### 이 단계의 역할
도달시간 600초(`TTI_URGENT`) 기준선과 실제 긴급 표적 분포 그래프를 대조해 임계값 설정의 적정성을 검증합니다.

#### 해야 할 일

- 미션 3에서 생성된 TTI 분포 및 순위 시각화 그래프에서 600초 기준선 위아래에 놓인 실제 위협·비위협 표적을 확인한다.
- AI에게 상위 10개 순위 일치율과 600초 기준선 경계에 위치한 고속 순항형(`fast_type`) 표적의 특성 요약을 요청한다.
- 그래프 해석 결과와 기본 추론 결과 보고서 핵심 내용을 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "TTI 600초 긴급 기준선 대비 실제 위협/비위협 분포와 상위 10개 순위 일치율을 해석하여 기본 추론 결과 보고서의 핵심 결론 문단 초안을 작성해줘."

  - **AI 결과 검증 포인트:** AI가 작성한 보고서 초안에 고속 순항형(`fast_type`, 150~250 m/s)은 거리가 멀어도 TTI가 짧아 600초 이내로 급격히 진입한다는 전술적 의미가 정확히 반영되었는지 확인합니다.

#### 핵심 확인 지표

> 도달시간 600초 기준선 대비 긴급 판정 정확도 90.0%, 기본 추론 결과 보고서 완성

#### 학습자가 제출해야 하는 내용

- 600초 기준선 대조 그래프 해석 및 기본 추론 결과 보고서
- AI 작성 보고서 팩트체크 및 수정 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`TTI_URGENT` 임계값을 `600`초에서 `300`초로 단축했을 때 정확도·재현율·오경보율 변화를 측정하고 임계값 선택의 장단점을 비교합니다.

#### 해야 할 일

- `TTI_URGENT`를 `600`에서 `300`으로 변경해 `baseline_rank()`를 재실행하고 정확도·미탐·오경보 변화를 기록한다.
- AI에게 임계값 `600초 vs 300초` 변경 시 놓치는 위협(False Negative)과 오경보 감소(False Positive) 간의 전술적 트레이드오프 비교표를 요청한다.
- 작전 상황(방공 요격 반응시간 확보 vs 경보 피로도 감소)에 따른 적정 임계값 판단 근거를 작성한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "TTI_URGENT를 600초에서 300초로 줄였을 때 정확도, 긴급 표적 미탐 건수, 오경보 건수 변화를 비교하는 실험 코드를 작성하고, 방공 작전 관점에서 두 임계값의 장단점을 정리해줘."

  - **AI 결과 검증 포인트:** 임계값을 300초로 낮출 때 오경보는 줄더라도 실제 긴급 위협을 뒤늦게 인지하는 미탐 위험이 커짐을 정량 수치로 확인하고, AI가 미탐 위험을 과소평가하지 않았는지 검증합니다.

#### 핵심 확인 지표

> TTI_URGENT 600초 vs 300초 정확도·오경보·미탐 비교표 작성

#### 학습자가 제출해야 하는 내용

- TTI_URGENT 변경 전후 성능 비교표 및 전술적 장단점 분석 메모
- AI 질의 프롬프트 및 미탐 리스크 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건
- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성
- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성
- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교
- AI 검증 기준: CPA·TTI 기하 연산의 방위각·부호 정합성을 AI와 교차 검산하고 기만체 오경보 한계 및 TTI 임계값(600초 vs 300초) 분석 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
"""가상 방어자산까지의 최근접 거리와 잔여 도달 예상시간(TTI), 접근 속도만으로 1차 위협 순위를 정렬하는 Baseline 랭킹 알고리즘을 구현하고 긴급 판정 정확도를 측정한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

# 판단 기준(투명성): ① 비행 경로가 가상 자산 반경 1.5 km 안을 지나는가(CPA)
#                   ② 그 자산까지 도달 예상시간(TTI)이 짧은가  ③ TTI 600초 이내면 '긴급'
raid = make_raid(7, 80)
rank = baseline_rank(raid).merge(raid[["id", "type", "spd", "urgent"]], on="id")
_, tti = geometry(raid)
dist = np.hypot(raid["x"], raid["y"])
y, p = rank["urgent"].to_numpy(), rank["urgent_pred"].to_numpy()
acc = (y == p).mean()
print(f"[Baseline 1차 위협 순위] 가상 비행체 {len(raid)}개, 실제 긴급 {y.sum()}개, 긴급 판정 {p.sum()}개")
top = rank.head(10).copy()
top["순위"] = range(1, 11)
top["tti_s"] = top["tti_s"].round(0); top["spd"] = top["spd"].round(0)
print(top[["순위", "id", "tti_s", "spd", "type", "urgent"]].rename(columns={"tti_s": "도달예상(s)", "spd": "속도(m/s)", "type": "실제유형", "urgent": "실제긴급"}).to_string(index=False))

print("\n[유형별 판정 결과] (긴급 판정 수 / 전체)")
for t, g in rank.groupby("type"):
    print(f"  {t:10s}: {int(g['urgent_pred'].sum())}/{len(g)}  (실제 긴급 {int(g['urgent'].sum())})")
top10_true = set(raid.sort_values("true_prio", ascending=False).head(10)["id"])
overlap = len(top10_true & set(rank.head(10)["id"])) / 10
print(f"\n긴급 판정 정확도 {acc:.1%} (목표 75% 이상) -> {'달성' if acc >= 0.75 else '미달'}")
print(f"상위 10개 순위 일치율 {overlap:.0%}, 오경보 {int((p & ~y).sum())}건(기만체·조류 등 유형 판별 없음이 원인)")
print("한계: 자산 가치와 비행체 유형을 고려하지 않으므로 미션 4에서 고도화한다. 순위는 '권고'이며 판단은 지휘관이 한다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(raid["x"], raid["y"], c=np.where(rank.set_index("id").loc[raid["id"], "urgent_pred"], "red", "gray"), s=18)
ax[0].scatter(ASSETS["x"], ASSETS["y"], marker="^", s=120, c="blue", label="virtual assets")
ax[0].set_title("Baseline: red = urgent (TTI <= 600 s)"); ax[0].set_xlabel("East (km)"); ax[0].set_ylabel("North (km)"); ax[0].legend()
ax[0].set_aspect("equal")
t15 = rank.head(15)
ax[1].barh(t15["id"][::-1], t15["tti_s"][::-1], color=["red" if u else "gray" for u in t15["urgent"][::-1]])
ax[1].axvline(TTI_URGENT, ls="--", c="k"); ax[1].set_title("Top-15 by TTI (red = truly urgent)"); ax[1].set_xlabel("TTI (s)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[Baseline 1차 위협 순위] 가상 비행체 80개, 실제 긴급 31개, 긴급 판정 39개
 순위     id  도달예상(s)  속도(m/s)      실제유형  실제긴급
  1 T07061     42.0     59.0     decoy False
  2 T07055     53.0     55.0  uav_type  True
  3 T07048     68.0    221.0 fast_type  True
  4 T07014    103.0    231.0 fast_type  True
  5 T07004    152.0    224.0 fast_type  True
  6 T07038    158.0     59.0  uav_type  True
  7 T07019    162.0     54.0  uav_type  True
  8 T07033    173.0     45.0  uav_type  True
  9 T07065    176.0    154.0 fast_type  True
 10 T07062    180.0    158.0 fast_type  True

[유형별 판정 결과] (긴급 판정 수 / 전체)
  bird      : 0/6  (실제 긴급 0)
  civil     : 1/4  (실제 긴급 0)
  decoy     : 7/16  (실제 긴급 0)
  fast_type : 13/13  (실제 긴급 13)
  uav_type  : 18/41  (실제 긴급 18)

긴급 판정 정확도 90.0% (목표 75% 이상) -> 달성
상위 10개 순위 일치율 80%, 오경보 8건(기만체·조류 등 유형 판별 없음이 원인)
한계: 자산 가치와 비행체 유형을 고려하지 않으므로 미션 4에서 고도화한다. 순위는 '권고'이며 판단은 지휘관이 한다.
```
</details>
